# LeetCode 592 — Fraction Addition and Subtraction bằng Python

Link đề: https://leetcode.com/problems/fraction-addition-and-subtraction/ · Độ khó: **Medium** · Chủ đề: Math, String, Greatest Common Divisor (ước chung lớn nhất), Simulation (mô phỏng)

Skeleton LeetCode:

```python
class Solution:
    def fractionAddition(self, expression: str) -> str:
        ...
```

Ghi chú: bài không có thuật toán "cao siêu", nhưng gom đủ các kỹ năng quan trọng: **tách chuỗi (parsing)**, **số học phân số**, **ước chung lớn nhất (gcd)** và **chọn mẫu số chung**. Notebook đi từ cách cộng ngây thơ (nhân hết các mẫu số) -> rút gọn từng bước -> **mẫu số cố định 2520** -> dùng thư viện `fractions`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: biểu thức chỉ có phân số, `+` và `-`; kết quả phải là **phân số tối giản**, số nguyên phải viết dạng `n/1`, số 0 viết `0/1`.
2. Nắm các công cụ Python cần thiết: `math.gcd` / `math.lcm`, phép chia lấy phần nguyên `//` với số âm, `str.split`, `str.replace`, `str.isdigit`, `re.findall`, f-string và `fractions.Fraction`.
3. Hiểu công thức cộng phân số `a/b + c/d = (a*d + c*b) / (b*d)` và cách đưa mọi phân số về **cùng một mẫu số** để chỉ cộng các số nguyên.
4. Đi qua 4 cách: cộng nhân hết mẫu số rồi rút gọn cuối, rút gọn sau mỗi bước, dùng mẫu số cố định `lcm(1..10) = 2520`, dùng `Fraction`.
5. Tránh các lỗi kinh điển: đọc từng ký tự giả định số một chữ số (`10/3` sẽ hỏng), tách chuỗi bằng `split('-')` làm mất dấu, dùng `/` ra số thực, quên rút gọn hoặc quên viết `0/1`, so sánh số thực.

## 2. Đề bài & ví dụ

Cho chuỗi `expression` biểu diễn một biểu thức cộng/trừ các phân số. Trả về kết quả dưới dạng chuỗi.

Kết quả phải là **phân số tối giản** (irreducible fraction). Nếu kết quả là số nguyên thì viết dưới dạng phân số có mẫu số `1`, ví dụ `2` phải đổi thành `2/1`.

**Ví dụ 1**

- Input: `expression = "-1/2+1/2"`
- Output: `"0/1"`

**Ví dụ 2**

- Input: `expression = "-1/2+1/2+1/3"`
- Output: `"1/3"`

**Ví dụ 3**

- Input: `expression = "1/3-1/2"`
- Output: `"-1/6"`

**Ràng buộc (theo đề):**

- Chuỗi chỉ chứa `'0'`..`'9'`, `'/'`, `'+'`, `'-'`; kết quả cũng vậy.
- Mỗi phân số (đầu vào và đầu ra) có dạng `±tử/mẫu`. Nếu phân số đầu tiên của đầu vào hoặc kết quả là **dương** thì dấu `+` được **bỏ**.
- Đầu vào chỉ gồm các phân số **tối giản hợp lệ**, tử và mẫu của mỗi phân số nằm trong `[1, 10]` (mẫu `1` nghĩa là một số nguyên).
- Số phân số trong khoảng `[1, 10]`.
- Tử và mẫu của **kết quả cuối** đảm bảo vừa số nguyên 32 bit.

Chú ý: tử/mẫu có thể là **hai chữ số** (`10`), nên không thể giả định mỗi số chỉ một ký tự.

In [1]:
from typing import List
from math import gcd, lcm
from fractions import Fraction
from functools import reduce
import re, random, time

examples = [
    ("-1/2+1/2", "0/1"),
    ("-1/2+1/2+1/3", "1/3"),
    ("1/3-1/2", "-1/6"),
]
for expr, expected in examples:
    print(f"{expr!r:18} len={len(expr):2} -> kỳ vọng {expected!r}")

'-1/2+1/2'         len= 8 -> kỳ vọng '0/1'
'-1/2+1/2+1/3'     len=12 -> kỳ vọng '1/3'
'1/3-1/2'          len= 7 -> kỳ vọng '-1/6'


## 3. Kiến thức nền

### 3.1. Ước chung lớn nhất `gcd` và bội chung nhỏ nhất `lcm`

`gcd(a, b)` là số nguyên dương lớn nhất chia hết cả `a` và `b`. `lcm(a, b)` là số nguyên dương nhỏ nhất chia hết cho cả hai. Python có sẵn `math.gcd` và `math.lcm` (nhận nhiều đối số từ Python 3.9).

Ba tính chất cần nhớ cho bài này:

- `gcd` luôn **không âm**: `gcd(-12, 18) = 6`.
- `gcd(0, n) = n`. Đặc biệt `gcd(0, 2520) = 2520`, điều này làm kết quả `0` tự rút về `0/1`.
- Một phân số `x/y` rút gọn bằng cách chia cả hai cho `gcd(x, y)`.

In [2]:
print(gcd(12, 18), gcd(-12, 18), gcd(0, 2520), gcd(7, 5))
print(lcm(4, 6), lcm(2, 3, 5), lcm(*range(1, 11)))      # lcm(1..10) = 2520

x, y = 840, 2520
g = gcd(x, y)
print(f"{x}/{y} rút gọn thành {x // g}/{y // g}  (chia cho {g})")
x, y = -840, 2520
g = gcd(x, y)
print(f"{x}/{y} rút gọn thành {x // g}/{y // g}  (g vẫn dương nên dấu nằm ở tử)")

6 6 2520 1
12 30 2520
840/2520 rút gọn thành 1/3  (chia cho 840)
-840/2520 rút gọn thành -1/3  (g vẫn dương nên dấu nằm ở tử)


### 3.2. Cộng hai phân số

Công thức tổng quát: `a/b + c/d = (a*d + c*b) / (b*d)`. Công thức này luôn đúng nhưng mẫu số có thể phình ra (`b*d`). Nếu hai phân số đã có **cùng mẫu số** thì chỉ việc cộng tử.

Ví dụ `1/3 - 1/2`: viết `-1/2` là `a = -1, b = 2` (dấu đi theo tử). `x/y = 1/3`, cộng: `(1*2 + (-1)*3) / (3*2) = -1/6`.

In [3]:
def add_frac(a, b, c, d):
    """a/b + c/d = (a*d + c*b)/(b*d), trả về (tử, mẫu) chưa rút gọn."""
    return a * d + c * b, b * d

print(add_frac(1, 3, -1, 2))                 # (-1, 6)
print(add_frac(1, 2, 1, 2))                  # (4, 4) -> chưa rút gọn, thực ra là 1
n, d = add_frac(1, 2, 1, 2)
print(f"{n // gcd(n, d)}/{d // gcd(n, d)}")  # 1/1

(-1, 6)
(4, 4)
1/1


### 3.3. Chia lấy phần nguyên `//` với số âm

Trong Python `//` **làm tròn xuống** (về phía âm vô cực), không phải cắt phần thập phân. `-7 // 2 = -4`, còn `int(-7 / 2) = -3`. Khi chia **chính xác** (không dư) thì `//` an toàn: `-840 // 840 = -1`. Bài này chỉ dùng `//` ở chỗ chia hết (`2520 // b`, `x // gcd`), nhưng bạn phải biết quy luật để không rơi vào bẫy ở bài khác.

In [4]:
print(-7 // 2, int(-7 / 2), 7 // 2)          # -4 -3 3
print(-7 % 2, 7 % -2)                        # 1 -1  (dấu của % theo số chia)
print(-840 // 840, 2520 // 8, 2520 % 7)      # -1 315 0
print("float:", 6 / 3, type(6 / 3), "| int:", 6 // 3, type(6 // 3))   # / luôn ra float

-4 -3 3
1 -1
-1 315 0
float: 2.0 <class 'float'> | int: 2 <class 'int'>


### 3.4. Xử lý chuỗi: `isdigit`, cắt lát, `split`, `replace`

- `ch.isdigit()` trả `True` nếu `ch` là chữ số.
- `s[i:j]` lấy các ký tự từ `i` tới `j - 1`.
- `s.split('/')` tách theo `/`: `"10/3".split('/') = ['10', '3']`.
- `s.replace('-', '+-')` chèn thêm `+` trước mỗi dấu `-`. Mẹo này biến `1/3-1/2` thành `1/3+-1/2`; tách theo `+` được các hạng tử **đã mang dấu**: `['1/3', '-1/2']`.
- `int("-1")`, `int("+1")` đều hợp lệ (cả dấu cũng được hiểu).

In [5]:
print("10/3".split('/'), int("10"), int("-1"), int("+1"))
print("1/3-1/2".replace('-', '+-'))
print("1/3-1/2".replace('-', '+-').split('+'))
print("-1/2+1/3".replace('-', '+-').split('+'))     # có phần tử rỗng ở đầu vì bắt đầu bằng '+'
print([c.isdigit() for c in "1/-2"])
print("-1/2+1/3"[1:4])

['10', '3'] 10 -1 1
1/3+-1/2
['1/3', '-1/2']
['', '-1/2', '1/3']
[True, False, False, True]
1/2


### 3.5. Biểu thức chính quy `re.findall` (dùng ở Cách 1 và Cách 4)

`re.findall(pattern, s)` trả list mọi đoạn khớp mẫu. Mẫu `[+-]?\d+/\d+` nghĩa là: một dấu tùy chọn (`+` hoặc `-`), một hay nhiều chữ số, dấu `/`, một hay nhiều chữ số. Mỗi kết quả là một hạng tử đã gồm dấu.

In [6]:
pattern = r"[+-]?\d+/\d+"
print(re.findall(pattern, "-1/2+10/3-7/10"))
print(re.findall(pattern, "1/3"))

['-1/2', '+10/3', '-7/10']
['1/3']


### 3.6. `fractions.Fraction`

Lớp `Fraction` của thư viện chuẩn lưu phân số **chính xác** và tự rút gọn. `Fraction(a, b)` tạo `a/b`; cộng, trừ đều cho `Fraction` mới; `.numerator`, `.denominator` lấy tử/mẫu (mẫu luôn dương, dấu ở tử).

In [7]:
f = Fraction(1, 3) - Fraction(1, 2)
print(f, f.numerator, f.denominator)
print(Fraction(0, 5), Fraction(4, 4), Fraction(-6, 4))
print(Fraction(1, 2) + Fraction(1, 2) == 1)

-1/6 -1 6
0 1 -3/2
True


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Tính `-1/2 + 1/2 + 1/3` như học sinh:

1. `-1/2 + 1/2 = 0`.
2. `0 + 1/3 = 1/3`. Đã tối giản. Đáp án `1/3`.

Tính `1/3 - 1/2`:

1. Mẫu số chung của 3 và 2 là 6.
2. `1/3 = 2/6`, `1/2 = 3/6`.
3. `2/6 - 3/6 = -1/6`.

Hai việc lặp lại trong mọi bài: **(a) đưa về mẫu chung rồi cộng tử** và **(b) rút gọn ở cuối bằng gcd**. Việc còn lại là **đọc** biểu thức: tách thành các hạng tử có dấu, mỗi hạng tử có tử và mẫu.

In [8]:
# Máy tính tay: mẫu chung 6 cho 1/3 - 1/2
den = 6
num = 1 * (den // 3) - 1 * (den // 2)
print(f"{num}/{den}", "-> rút gọn:", f"{num // gcd(num, den)}/{den // gcd(num, den)}")

# Với ví dụ 2: mẫu chung 6 cho -1/2 + 1/2 + 1/3
num = -1 * (6 // 2) + 1 * (6 // 2) + 1 * (6 // 3)
g = gcd(num, 6)
print(f"{num}/6 -> {num // g}/{6 // g}")

-1/6 -> rút gọn: -1/6
2/6 -> 1/3


## 5. Cách 1 — Brute Force / baseline: cộng thẳng, nhân hết các mẫu số, rút gọn ở cuối

**Ý tưởng.** Làm đúng như công thức cộng hai phân số, không tối ưu gì: giữ tổng `x/y` (ban đầu `0/1`). Với mỗi hạng tử `a/b` (có dấu): `x = x*b + a*y`, `y = y*b`. Không rút gọn gì ở giữa; sau khi cộng hết, chia cả hai cho `gcd(x, y)`.

Dùng `re.findall` để tách hạng tử, vì vậy phần đọc chuỗi rất ngắn.

In [9]:
def fraction_naive(expression: str) -> str:
    x, y = 0, 1
    for term in re.findall(r"[+-]?\d+/\d+", expression):
        a, b = map(int, term.split('/'))
        x = x * b + a * y
        y *= b
    g = gcd(x, y)
    return f"{x // g}/{y // g}"

for expr, expected in examples:
    got = fraction_naive(expr)
    print(expr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

-1/2+1/2 -> 0/1 OK
-1/2+1/2+1/3 -> 1/3 OK
1/3-1/2 -> -1/6 OK


**Độ phức tạp và nút thắt.** Với `k` hạng tử, thời gian `O(k)` phép tính + tách chuỗi, rất nhanh. Nhưng mẫu số `y` là **tích mọi mẫu số**: tới 10 phân số mẫu 10 thì `y = 10^10`, vượt 32 bit (`2^31 - 1 = 2.147.483.647`). Python có số nguyên lớn tùy ý nên vẫn chạy đúng, nhưng ở Java/C++ sẽ **tràn số**. Đây là lý do ta cần các cách giữ số nhỏ.

In [10]:
expr_worst = "+".join(["1/10"] * 10)
x, y = 0, 1
for term in re.findall(r"[+-]?\d+/\d+", expr_worst):
    a, b = map(int, term.split('/'))
    x, y = x * b + a * y, y * b
print(expr_worst)
print("trước khi rút gọn:", f"{x}/{y}", "| vượt 32 bit?", y > 2**31 - 1)
print("sau khi rút gọn  :", fraction_naive(expr_worst))

1/10+1/10+1/10+1/10+1/10+1/10+1/10+1/10+1/10+1/10
trước khi rút gọn: 10000000000/10000000000 | vượt 32 bit? True
sau khi rút gọn  : 1/1


## 6. Ý tưởng thuật toán / bước đột phá

Hai cách giữ số nhỏ:

- **Rút gọn sau mỗi bước.** Sau mỗi lần cộng, chia `x`, `y` cho `gcd(x, y)`. Khi đó `y` luôn là mẫu số của một phân số tối giản, tức là ước của `lcm` các mẫu đã gặp, không phình ra theo tích.
- **Mẫu số cố định.** Mọi mẫu số trong đầu vào nằm trong `[1, 10]`. Vậy `lcm(1, 2, ..., 10) = 2520` chia hết cho **mọi** mẫu số có thể. Mỗi `a/b` viết được thành `a*(2520/b) / 2520` với tử là **số nguyên**. Cộng cả biểu thức chỉ là cộng các số nguyên rồi **rút gọn đúng một lần**.

Cách thứ hai là cách gọn nhất, và cũng cho phép đọc biểu thức bằng một dòng `replace/split`.

| Cách | Ý tưởng |
|---|---|
| 2 | Đọc từng ký tự bằng tay, rút gọn sau mỗi hạng tử |
| 3 | Mẫu số cố định 2520, đọc bằng `replace('-', '+-').split('+')`, rút gọn một lần |
| 4 | Giao cho `Fraction` làm hết |

In [11]:
print("lcm(1..10) =", lcm(*range(1, 11)))
print("mọi mẫu 1..10 đều chia hết 2520?", all(2520 % b == 0 for b in range(1, 11)))
print("còn 2520 // b:", {b: 2520 // b for b in range(1, 11)})
print("so với tích 6*7*8*9*10 =", 6 * 7 * 8 * 9 * 10, "(cũng chia hết mọi mẫu 1..10:", all(30240 % b == 0 for b in range(1, 11)), ")")

lcm(1..10) = 2520
mọi mẫu 1..10 đều chia hết 2520? True
còn 2520 // b: {1: 2520, 2: 1260, 3: 840, 4: 630, 5: 504, 6: 420, 7: 360, 8: 315, 9: 280, 10: 252}
so với tích 6*7*8*9*10 = 30240 (cũng chia hết mọi mẫu 1..10: True )


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đọc từng ký tự bằng tay + rút gọn sau mỗi hạng tử

**Giải thích từng đoạn.**

- Giữ tổng `x/y` (khởi tạo `0/1`) và con trỏ `i` chạy trên chuỗi.
- Tại mỗi hạng tử: nếu `expression[i]` là `+` hoặc `-` thì ghi lại dấu và nhảy qua; nếu không (hạng tử đầu tiên không dấu) thì dấu là `+`.
- Tử số: từ `i` tới khi gặp `/`. Mẫu số: sau `/` tới khi hết chữ số. Dùng cắt lát và `int`, nên hai chữ số như `10` được xử lý đúng.
- Cộng: `x = x*b + sign*a*y`, `y = y*b`, rồi chia cả hai cho `gcd(x, y)` (`gcd(0, y) = y` nên `0` tự về `0/1`).

In [12]:
def fraction_scan(expression: str) -> str:
    x, y = 0, 1
    i, n = 0, len(expression)
    while i < n:
        sign = 1
        if expression[i] in "+-":
            if expression[i] == "-":
                sign = -1
            i += 1
        j = i
        while expression[j] != "/":          # tử số: tới dấu '/'
            j += 1
        a = int(expression[i:j])
        j += 1
        k = j
        while k < n and expression[k].isdigit():   # mẫu số: các chữ số liên tiếp
            k += 1
        b = int(expression[j:k])
        x = x * b + sign * a * y
        y *= b
        g = gcd(x, y)
        x //= g
        y //= g
        i = k
    return f"{x}/{y}"

for expr, expected in examples:
    got = fraction_scan(expr)
    print(expr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected
print(fraction_scan("10/3-7/10"))      # có số hai chữ số

-1/2+1/2 -> 0/1 OK
-1/2+1/2+1/3 -> 1/3 OK
1/3-1/2 -> -1/6 OK
79/30


**Code nộp LeetCode (Cách 2):**

In [13]:
class Solution:
    def fractionAddition(self, expression: str) -> str:
        from math import gcd
        x, y = 0, 1
        i, n = 0, len(expression)
        while i < n:
            sign = 1
            if expression[i] in "+-":
                if expression[i] == "-":
                    sign = -1
                i += 1
            j = i
            while expression[j] != "/":
                j += 1
            a = int(expression[i:j])
            j += 1
            k = j
            while k < n and expression[k].isdigit():
                k += 1
            b = int(expression[j:k])
            x = x * b + sign * a * y
            y *= b
            g = gcd(x, y)
            x //= g
            y //= g
            i = k
        return f"{x}/{y}"

sol_scan = Solution().fractionAddition
print([sol_scan(e) for e, _ in examples])

['0/1', '1/3', '-1/6']


**Độ phức tạp (Cách 2):** thời gian `O(L)` với `L = len(expression)` (mỗi ký tự thăm một lần) cộng `O(k log M)` cho các lần gcd (`M` là cỡ số, rất nhỏ); bộ nhớ `O(1)` (chỉ vài biến, không tạo list).

---

### 7.2. Cách 3 — Mẫu số cố định 2520, một lần rút gọn

**Giải thích từng đoạn.**

- `base = 2520 = lcm(1..10)`: chia hết cho mọi mẫu số trong đề.
- `expression.replace('-', '+-').split('+')` tách các hạng tử đã mang dấu. Nếu chuỗi bắt đầu bằng `-` thì phần tử đầu rỗng (`''`), bỏ qua bằng `if not token: continue`.
- Mỗi hạng tử: `a, b = token.split('/')`; đóng góp vào tổng là `int(a) * (base // int(b))`. `int('-1')` đã mang dấu.
- Sau cùng `total / base`; chia cho `g = gcd(total, base)`.

In [14]:
def fraction_fixed(expression: str) -> str:
    base = 2520
    total = 0
    for token in expression.replace("-", "+-").split("+"):
        if not token:
            continue
        a, b = token.split("/")
        total += int(a) * (base // int(b))
    g = gcd(total, base)
    return f"{total // g}/{base // g}"

for expr, expected in examples:
    got = fraction_fixed(expr)
    print(expr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

-1/2+1/2 -> 0/1 OK
-1/2+1/2+1/3 -> 1/3 OK
1/3-1/2 -> -1/6 OK


**Code nộp LeetCode (Cách 3):**

In [15]:
class Solution:
    def fractionAddition(self, expression: str) -> str:
        from math import gcd
        base = 2520                       # lcm(1..10), chia hết mọi mẫu số trong đề
        total = 0
        for token in expression.replace("-", "+-").split("+"):
            if not token:
                continue
            a, b = token.split("/")
            total += int(a) * (base // int(b))
        g = gcd(total, base)
        return f"{total // g}/{base // g}"

sol_fixed = Solution().fractionAddition
print([sol_fixed(e) for e, _ in examples])

['0/1', '1/3', '-1/6']


**Độ phức tạp (Cách 3):** thời gian `O(L)`; bộ nhớ `O(L)` cho list các hạng tử (nhỏ: tối đa 10 phần tử). Mọi số trung gian đều bé: `|total| <= 10 * 10 * 2520 = 252000`.

---

### 7.3. Cách 4 — Dùng `fractions.Fraction`

**Ý tưởng.** Để thư viện lo việc cộng, rút gọn, chuẩn hóa dấu. Ta chỉ còn phải **đọc** các hạng tử, rồi in `numerator/denominator`.

**Giải thích từng đoạn.**

- `total = Fraction(0)`.
- Với mỗi hạng tử `a/b` (khớp bằng `re.findall`) cộng `Fraction(a, b)`. `Fraction` tự rút gọn sau mỗi phép cộng.
- Kết quả `0` có `denominator = 1` nên tự ra `0/1`.

In [16]:
def fraction_lib(expression: str) -> str:
    total = Fraction(0)
    for term in re.findall(r"[+-]?\d+/\d+", expression):
        a, b = map(int, term.split("/"))
        total += Fraction(a, b)
    return f"{total.numerator}/{total.denominator}"

for expr, expected in examples:
    got = fraction_lib(expr)
    print(expr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

-1/2+1/2 -> 0/1 OK
-1/2+1/2+1/3 -> 1/3 OK
1/3-1/2 -> -1/6 OK


**Code nộp LeetCode (Cách 4):**

In [17]:
class Solution:
    def fractionAddition(self, expression: str) -> str:
        import re
        from fractions import Fraction
        total = Fraction(0)
        for term in re.findall(r"[+-]?\d+/\d+", expression):
            a, b = map(int, term.split("/"))
            total += Fraction(a, b)
        return f"{total.numerator}/{total.denominator}"

sol_lib = Solution().fractionAddition
print([sol_lib(e) for e, _ in examples])

['0/1', '1/3', '-1/6']


**Độ phức tạp (Cách 4):** thời gian `O(L)` (cộng `Fraction` chậm hơn số nguyên một hằng số, nhưng `k <= 10`); bộ nhớ `O(L)` cho list kết quả `findall`.

Cách 4 gọn nhất nhưng trong phỏng vấn người ta thường muốn thấy bạn tự làm gcd/mẫu chung, nên hãy nắm vững Cách 2 và 3.

## 8. TRACE từng bước

### 8.1. Cách 3 với `"-1/2+1/2+1/3"`

Mẫu chung `2520`. Đổi mỗi hạng tử sang đơn vị `1/2520`:

| hạng tử | `a` | `b` | `2520 // b` | đóng góp `a * (2520 // b)` | `total` sau bước |
|---|---|---|---|---|---|
| `-1/2` | -1 | 2 | 1260 | -1260 | -1260 |
| `1/2` | 1 | 2 | 1260 | +1260 | 0 |
| `1/3` | 1 | 3 | 840 | +840 | 840 |

Cuối cùng `840 / 2520`, `gcd = 840` -> `1/3`. Hàm `trace_fixed` dưới đây in từng bước.

In [18]:
def trace_fixed(expression: str) -> str:
    base = 2520
    total = 0
    tokens = expression.replace("-", "+-").split("+")
    print("sau replace/split:", tokens)
    for token in tokens:
        if not token:
            print("  (bỏ phần tử rỗng)")
            continue
        a, b = token.split("/")
        a, b = int(a), int(b)
        contribution = a * (base // b)
        total += contribution
        print(f"  hạng tử {token:>6}: a={a:>3}, b={b:>2}, {base}//{b}={base // b:>4}, đóng góp {contribution:>6}, total={total}")
    g = gcd(total, base)
    print(f"kết quả thô {total}/{base}, gcd = {g} -> {total // g}/{base // g}")
    return f"{total // g}/{base // g}"

assert trace_fixed("-1/2+1/2+1/3") == "1/3"
print()
assert trace_fixed("1/3-1/2") == "-1/6"

sau replace/split: ['', '-1/2', '1/2', '1/3']
  (bỏ phần tử rỗng)
  hạng tử   -1/2: a= -1, b= 2, 2520//2=1260, đóng góp  -1260, total=-1260
  hạng tử    1/2: a=  1, b= 2, 2520//2=1260, đóng góp   1260, total=0
  hạng tử    1/3: a=  1, b= 3, 2520//3= 840, đóng góp    840, total=840
kết quả thô 840/2520, gcd = 840 -> 1/3

sau replace/split: ['1/3', '-1/2']
  hạng tử    1/3: a=  1, b= 3, 2520//3= 840, đóng góp    840, total=840
  hạng tử   -1/2: a= -1, b= 2, 2520//2=1260, đóng góp  -1260, total=-420
kết quả thô -420/2520, gcd = 420 -> -1/6


### 8.2. Cách 2 với `"1/3-1/2+1/6"`: rút gọn sau mỗi bước

| bước | hạng tử | `x/y` trước | tính `x*b + sign*a*y` / `y*b` | gcd | `x/y` sau |
|---|---|---|---|---|---|
| 1 | `+1/3` | 0/1 | `(0*3+1*1)/(1*3) = 1/3` | 1 | 1/3 |
| 2 | `-1/2` | 1/3 | `(1*2-1*3)/(3*2) = -1/6` | 1 | -1/6 |
| 3 | `+1/6` | -1/6 | `(-1*6+1*6)/(6*6) = 0/36` | 36 | 0/1 |

In [19]:
def trace_scan(expression: str) -> str:
    x, y = 0, 1
    i, n = 0, len(expression)
    step = 0
    while i < n:
        step += 1
        sign = 1
        if expression[i] in "+-":
            sign = -1 if expression[i] == "-" else 1
            i += 1
        j = i
        while expression[j] != "/":
            j += 1
        a = int(expression[i:j])
        j += 1
        k = j
        while k < n and expression[k].isdigit():
            k += 1
        b = int(expression[j:k])
        old = (x, y)
        x = x * b + sign * a * y
        y *= b
        g = gcd(x, y)
        print(f"bước {step}: đọc {'+' if sign > 0 else '-'}{a}/{b}; {old[0]}/{old[1]} -> {x}/{y} (gcd={g}) -> {x // g}/{y // g}")
        x //= g
        y //= g
        i = k
    return f"{x}/{y}"

assert trace_scan("1/3-1/2+1/6") == "0/1"

bước 1: đọc +1/3; 0/1 -> 1/3 (gcd=1) -> 1/3
bước 2: đọc -1/2; 1/3 -> -1/6 (gcd=1) -> -1/6
bước 3: đọc +1/6; -1/6 -> 0/36 (gcd=36) -> 0/1


### 8.3. Con trỏ khi đọc chuỗi `"-1/2+10/3"`

Xem `i`, `j`, `k` di chuyển ra sao khi có số hai chữ số `10`:

In [20]:
def trace_pointers(expression: str):
    i, n = 0, len(expression)
    while i < n:
        sign_char = ""
        if expression[i] in "+-":
            sign_char = expression[i]
            i += 1
        j = i
        while expression[j] != "/":
            j += 1
        a_text = expression[i:j]
        j += 1
        k = j
        while k < n and expression[k].isdigit():
            k += 1
        b_text = expression[j:k]
        print(f"dấu={sign_char or '(không)'!r:9} tử={a_text!r:5} (vị trí {i}..{j - 2})  mẫu={b_text!r:5} (vị trí {j}..{k - 1})  -> con trỏ kế tiếp = {k}")
        i = k

print("expression =", "-1/2+10/3", " độ dài", len("-1/2+10/3"))
trace_pointers("-1/2+10/3")

expression = -1/2+10/3  độ dài 9
dấu='-'       tử='1'   (vị trí 1..1)  mẫu='2'   (vị trí 3..3)  -> con trỏ kế tiếp = 4
dấu='+'       tử='10'  (vị trí 5..6)  mẫu='3'   (vị trí 8..8)  -> con trỏ kế tiếp = 9


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (mẫu số 2520): ngắn, không có số lớn, ít chỗ sai.

```python
class Solution:
    def fractionAddition(self, expression: str) -> str:
        from math import gcd
        base = 2520
        total = 0
        for token in expression.replace("-", "+-").split("+"):
            if not token:
                continue
            a, b = token.split("/")
            total += int(a) * (base // int(b))
        g = gcd(total, base)
        return f"{total // g}/{base // g}"
```

Việc `import gcd` nằm trong hàm để đoạn mã dán là tự đủ; bạn cũng có thể đặt `from math import gcd` ở đầu file.

In [21]:
class Solution:
    def fractionAddition(self, expression: str) -> str:
        from math import gcd
        base = 2520
        total = 0
        for token in expression.replace("-", "+-").split("+"):
            if not token:
                continue
            a, b = token.split("/")
            total += int(a) * (base // int(b))
        g = gcd(total, base)
        return f"{total // g}/{base // g}"

sol = Solution()
for expr, expected in examples:
    got = sol.fractionAddition(expr)
    print(expr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

-1/2+1/2 -> 0/1 OK
-1/2+1/2+1/3 -> 1/3 OK
1/3-1/2 -> -1/6 OK


## 10. Vì sao đúng

**(a) Cộng đúng.** Với `a/b` và mẫu chung `D` chia hết cho `b`: `a/b = a*(D/b) / D` vì nhân cả tử lẫn mẫu với `D/b`. `D = 2520` chia hết cho mọi `b` trong `[1, 10]` nên `D/b` là số nguyên, **không mất chính xác**. Tổng các phân số cùng mẫu `D` là tổng các tử trên `D`.

**(b) Dấu.** `replace('-', '+-')` làm mỗi dấu `-` thành `+-`, tức mỗi hạng tử trở thành `[dấu]a/b` và dấu thuộc về tử. `split('+')` không cắt nhầm vì không còn `+` nào ngoài các dấu phân cách (dấu `+` gốc cũng là phân cách). Phần tử rỗng chỉ xuất hiện ở đầu khi chuỗi bắt đầu bằng `-` (hoặc `+`), nên bỏ đi là đúng. Hạng tử đầu không dấu cũng không cần xử lý riêng.

**(c) Tối giản và dạng chuẩn.** Chia tử và mẫu cho `g = gcd(total, D)` cho phân số tối giản; mẫu `D/g > 0` vì `gcd` luôn dương, nên dấu nằm ở tử. Phân số tối giản với mẫu dương là **duy nhất**, nên khớp đáp án của đề. Khi `total = 0`: `gcd(0, D) = D`, kết quả `0/1` đúng yêu cầu. Khi `total` là bội của `D` (kết quả nguyên) ta được mẫu `1`.

**(d) Cách 2** duy trì bất biến "sau mỗi bước, `x/y` là tổng các hạng tử đã đọc và đã tối giản", theo công thức cộng hai phân số rồi chia cho gcd; bất biến giữ đúng nên kết quả cuối đúng.

Kiểm chứng bằng máy: so sánh với **oracle độc lập** — đổi biểu thức thành mã Python dùng `Fraction` rồi để chính Python `eval` (chỉ dùng trong notebook, không bao giờ `eval` dữ liệu không tin cậy).

In [22]:
def oracle(expression: str) -> str:
    py = re.sub(r"(\d+)", r"Fraction(\1)", expression)     # 1/3-1/2 -> Fraction(1)/Fraction(3)-Fraction(1)/Fraction(2)
    v = eval(py)
    return f"{v.numerator}/{v.denominator}"

print(oracle("1/3-1/2"), oracle("-1/2+1/2"), oracle("-1/2+1/2+1/3"))
for expr, expected in examples:
    assert oracle(expr) == expected

# tập mọi phân số tối giản hợp lệ trong đề
FR = [(a, b) for a in range(1, 11) for b in range(1, 11) if gcd(a, b) == 1]
print("số phân số tối giản hợp lệ (tử, mẫu trong 1..10):", len(FR))

def make_expr(terms):
    """terms: danh sách (dấu, a, b); dấu + ở hạng tử đầu bị bỏ."""
    out = []
    for idx, (s, a, b) in enumerate(terms):
        sign = "-" if s < 0 else ("" if idx == 0 else "+")
        out.append(f"{sign}{a}/{b}")
    return "".join(out)

print(make_expr([(1, 1, 3), (-1, 1, 2)]), make_expr([(-1, 1, 2), (1, 10, 3)]))

# 2 hạng tử: duyệt MỌI cặp có dấu và đối chiếu DP 3 cách với oracle
count = 0
for (a, b) in FR:
    for (c, d) in FR:
        for s1 in (1, -1):
            for s2 in (1, -1):
                e = make_expr([(s1, a, b), (s2, c, d)])
                ref = oracle(e)
                assert fraction_fixed(e) == ref and fraction_scan(e) == ref, e
                count += 1
print("đã đối chiếu", count, "biểu thức 2 hạng tử: OK")

-1/6 0/1 1/3
số phân số tối giản hợp lệ (tử, mẫu trong 1..10): 63
1/3-1/2 -1/2+10/3


đã đối chiếu 15876 biểu thức 2 hạng tử: OK


## 11. Time / Space Complexity

Gọi `L = len(expression)` (cỡ 10 hạng tử x tối đa 6 ký tự = tối đa khoảng 60) và `k <= 10` là số hạng tử.

| Cách | Time | Space | Số lớn nhất |
|---|---|---|---|
| 1. Nhân hết mẫu, rút gọn cuối | `O(L)` | `O(L)` | tới `10^10` (tràn 32 bit ở Java/C++) |
| 2. Quét tay, rút gọn mỗi bước | `O(L + k log M)` | `O(1)` | mẫu <= 2520 |
| 3. Mẫu cố định 2520 | `O(L)` | `O(L)` | tổng tuyệt đối `<= 252000` |
| 4. `Fraction` | `O(L)` | `O(L)` | tự quản lý |

Vì `L <= 60` nên mọi cách đều chạy tức thì; khác biệt nằm ở sự **an toàn số học** và độ gọn.

In [23]:
random.seed(592)
big_exprs = [make_expr([(random.choice((1, -1)), *random.choice(FR)) for _ in range(10)]) for _ in range(20000)]
print("ví dụ biểu thức dài nhất:", max(big_exprs, key=len), "| độ dài lớn nhất:", max(map(len, big_exprs)))
for name, fn in [("naive", fraction_naive), ("scan", fraction_scan), ("fixed", fraction_fixed), ("Fraction", fraction_lib)]:
    t0 = time.perf_counter()
    for e in big_exprs:
        fn(e)
    dt = (time.perf_counter() - t0) * 1000
    print(f"{name:9} 20000 biểu thức: {dt:7.1f} ms")

ví dụ biểu thức dài nhất: -10/1+9/10+3/10+7/10+10/1-1/9-10/3+5/8+1/1-3/10 | độ dài lớn nhất: 47


naive     20000 biểu thức:   132.7 ms


scan      20000 biểu thức:   119.0 ms
fixed     20000 biểu thức:    71.6 ms


Fraction  20000 biểu thức:   456.9 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ (cách làm sai bị lộ), đối chiếu tất cả các cách với oracle trên nhiều input và ngẫu nhiên có seed cố định (đúng ràng buộc: 1..10 phân số tối giản, tử/mẫu trong 1..10).

In [24]:
solutions = {
    "naive": fraction_naive,
    "scan": fraction_scan,
    "fixed": fraction_fixed,
    "lib": fraction_lib,
    "final": Solution().fractionAddition,
}

# 12.1 ví dụ đề
for expr, expected in examples:
    for name, fn in solutions.items():
        assert fn(expr) == expected, (name, expr)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay)
edge = [
    ("1/3", "1/3"),                         # một hạng tử
    ("-1/3", "-1/3"),                       # một hạng tử âm
    ("5/1", "5/1"),                         # số nguyên giữ dạng n/1
    ("-5/1", "-5/1"),
    ("1/2+1/2", "1/1"),                     # kết quả nguyên dương
    ("1/2-1/2", "0/1"),                     # kết quả 0
    ("-1/2+1/2", "0/1"),
    ("+".join(["10/1"] * 10), "100/1"),    # lớn nhất
    ("-" + "-".join(["10/1"] * 10), "-100/1"),               # nhỏ nhất
    ("+".join(["1/10"] * 10), "1/1"),                         # mười lần 1/10
    ("10/3-7/10", "79/30"),                 # số hai chữ số
    ("1/2+1/3+1/4+1/5+1/6+1/7+1/8+1/9+1/10", "4861/2520"),  # dùng hết mẫu 2520
    ("1/9-1/9+1/9", "1/9"),
    ("3/10+7/10", "1/1"),
    ("9/10-1/10", "4/5"),                   # kết quả cần rút gọn: 8/10 -> 4/5
]
for expr, expected in edge:
    assert oracle(expr) == expected, ("oracle sai kỳ vọng", expr, oracle(expr), expected)
    for name, fn in solutions.items():
        assert fn(expr) == expected, (name, expr, fn(expr), expected)
print("12.2 edge case: OK", len(edge), "ca")

# 12.3 phản ví dụ: các cách làm sai
def wrong_float(expression: str) -> str:
    total = 0.0
    for term in re.findall(r"[+-]?\d+/\d+", expression):
        a, b = map(int, term.split("/"))
        total += a / b
    f = Fraction(total).limit_denominator(10)     # cố đoán lại phân số
    return f"{f.numerator}/{f.denominator}"

def wrong_single_digit(expression: str) -> str:
    # giả định tử và mẫu đều một chữ số
    expr = expression if expression[0] in "+-" else "+" + expression
    total = 0
    for i in range(0, len(expr), 4):
        sign = -1 if expr[i] == "-" else 1
        total += sign * int(expr[i + 1]) * (2520 // int(expr[i + 3]))
    g = gcd(total, 2520)
    return f"{total // g}/{2520 // g}"

print("float: 1/10+1/5-3/10 ->", 1/10 + 1/5 - 3/10, "(đáng ra phải đúng bằng 0)")
assert 1/10 + 1/5 - 3/10 != 0
print("một chữ số: '10/3-7/10' ->", end=" ")
try:
    print(wrong_single_digit("10/3-7/10"))
except Exception as ex:
    print("lỗi", type(ex).__name__)
print("12.3 phản ví dụ: OK (các cách sai bị lộ)")

# 12.4 ngẫu nhiên có seed, đúng ràng buộc
random.seed(2026)
for _ in range(3000):
    k = random.randint(1, 10)
    terms = [(random.choice((1, -1)), *random.choice(FR)) for _ in range(k)]
    e = make_expr(terms)
    ref = oracle(e)
    for name, fn in solutions.items():
        assert fn(e) == ref, (name, e, fn(e), ref)
    num, den = map(int, ref.split("/"))
    assert den > 0 and gcd(abs(num), den) == 1 and abs(num) < 2**31 and den < 2**31
print("12.4 ngẫu nhiên 3000 biểu thức (k = 1..10): OK")

# 12.5 mọi bộ ba mẫu số: các phân số 1/b với b trong 1..10
for b1 in range(1, 11):
    for b2 in range(1, 11):
        for b3 in range(1, 11):
            e = f"1/{b1}-1/{b2}+1/{b3}"
            assert fraction_fixed(e) == oracle(e) == fraction_scan(e) == fraction_lib(e) == fraction_naive(e), e
print("12.5 mọi bộ ba mẫu số 1..10: OK")

12.1 ví dụ đề: OK
12.2 edge case: OK 15 ca
float: 1/10+1/5-3/10 -> 5.551115123125783e-17 (đáng ra phải đúng bằng 0)
một chữ số: '10/3-7/10' -> lỗi ValueError
12.3 phản ví dụ: OK (các cách sai bị lộ)


12.4 ngẫu nhiên 3000 biểu thức (k = 1..10): OK
12.5 mọi bộ ba mẫu số 1..10: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Giả định mỗi số chỉ có một chữ số

Code quét theo bước cố định kiểu `expr[i]`, `expr[i + 2]` chạy đúng ví dụ của đề nhưng gãy khi có `10`.

In [25]:
def wrong_fixed_width(expression: str) -> str:
    expr = expression if expression[0] in "+-" else "+" + expression
    total, i = 0, 0
    while i < len(expr):
        sign = -1 if expr[i] == "-" else 1
        a, b = int(expr[i + 1]), int(expr[i + 3])     # chỉ lấy đúng 1 ký tự mỗi số
        total += sign * a * (2520 // b)
        i += 4                                         # giả định mỗi hạng tử dài 4 ký tự
    g = gcd(total, 2520)
    return f"{total // g}/{2520 // g}"

print("đề cho:", wrong_fixed_width("1/3-1/2"), "(đúng)")
try:
    print("10/3-7/10:", wrong_fixed_width("10/3-7/10"))
except Exception as ex:
    print("10/3-7/10 gây lỗi:", type(ex).__name__, ex)
print("đúng phải là:", fraction_fixed("10/3-7/10"))
assert wrong_fixed_width("1/3-1/2") == "-1/6"

đề cho: -1/6 (đúng)
10/3-7/10 gây lỗi: ValueError invalid literal for int() with base 10: '/'
đúng phải là: 79/30


Vì sao sai: mẫu hay tử là `10` thì dài 2 ký tự; con trỏ lệch và đọc nhầm ký tự `/` hay dấu. **Đúng:** luôn đọc "tới khi gặp `/`" hoặc "tới khi hết chữ số" như Cách 2, hoặc tách bằng `split` như Cách 3.

### Lỗi 2 — Tách bằng `split('-')` hoặc `split('+')` mất dấu

Tách theo `+` rồi tách tiếp theo `-` làm ta mất thông tin hạng tử nào mang dấu gì.

In [26]:
expr = "1/3-1/2+1/6"
print("split('+')  :", expr.split("+"))          # ['1/3-1/2', '1/6'] -> còn dính dấu '-' bên trong
print("split('-')  :", expr.split("-"))          # ['1/3', '1/2+1/6'] -> mất dấu '-'
print("đúng (replace) :", expr.replace("-", "+-").split("+"))
assert expr.replace("-", "+-").split("+") == ["1/3", "-1/2", "1/6"]

split('+')  : ['1/3-1/2', '1/6']
split('-')  : ['1/3', '1/2+1/6']
đúng (replace) : ['1/3', '-1/2', '1/6']


Vì sao sai: `split` **xóa** dấu phân cách. **Đúng:** chèn `+` trước mỗi `-` rồi tách theo `+` (các dấu `-` vẫn ở lại trong hạng tử).

### Lỗi 3 — Dùng `/` (chia thực) rồi in ra số thực

`x += sign * a * y / b` làm `x` thành `float`; kết quả in ra là `"840.0/2520.0"`.

In [27]:
def wrong_true_div(expression: str) -> str:
    y = 2520
    x = 0
    for token in expression.replace("-", "+-").split("+"):
        if not token:
            continue
        a, b = token.split("/")
        x += int(a) * y / int(b)            # SAI: '/' ra float
    g = gcd(int(x), y)
    return f"{x // g}/{y // g}"

print(wrong_true_div("-1/2+1/2+1/3"), "<- có '.0' nên không khớp chuỗi đáp án")
assert wrong_true_div("-1/2+1/2+1/3") != "1/3"
print("đúng:", fraction_fixed("-1/2+1/2+1/3"))

1.0/3 <- có '.0' nên không khớp chuỗi đáp án
đúng: 1/3


Vì sao sai: `/` trong Python 3 luôn cho `float`, nên chuỗi kết quả chứa `.0` và đôi khi có sai số làm tròn. **Đúng:** dùng `//` khi chia hết (như `2520 // b`).

### Lỗi 4 — Quên rút gọn hoặc quên dạng `0/1`

Trả `f"{total}/{base}"` mà không chia gcd, hoặc trả `"0"` khi tổng bằng 0.

In [28]:
def wrong_no_reduce(expression: str) -> str:
    total = 0
    for token in expression.replace("-", "+-").split("+"):
        if token:
            a, b = token.split("/")
            total += int(a) * (2520 // int(b))
    return f"{total}/2520"

def wrong_zero(expression: str) -> str:
    r = fraction_fixed(expression)
    return "0" if r.startswith("0/") else r

print("không rút gọn:", wrong_no_reduce("-1/2+1/2+1/3"), "| đúng:", fraction_fixed("-1/2+1/2+1/3"))
print("kết quả 0    :", wrong_zero("-1/2+1/2"), "| đúng:", fraction_fixed("-1/2+1/2"))
assert wrong_no_reduce("-1/2+1/2+1/3") != "1/3"
assert wrong_zero("-1/2+1/2") != "0/1"

không rút gọn: 840/2520 | đúng: 1/3
kết quả 0    : 0 | đúng: 0/1


Vì sao sai: đề yêu cầu **tối giản** và số nguyên phải có `/1`. **Đúng:** luôn `g = gcd(total, base)` rồi chia cả hai (với `total = 0` thì `g = base`, tự ra `0/1`).

### Lỗi 5 — Tính bằng số thực rồi so sánh/đoán lại phân số

Số thực không biểu diễn chính xác `1/10`, nên `1/10 + 1/5 - 3/10` không đúng bằng 0, và đoán lại phân số từ số thực dễ sai.

In [29]:
print(1/10 + 1/5 - 3/10)                       # 5.55e-17, không phải 0
print(0.1 + 0.2 == 0.3)                        # False
print("dùng float rồi đoán lại:", wrong_float("1/10+1/5-3/10"), "| đúng:", fraction_fixed("1/10+1/5-3/10"))
assert fraction_fixed("1/10+1/5-3/10") == "0/1"

5.551115123125783e-17
False
dùng float rồi đoán lại: 0/1 | đúng: 0/1


Vì sao sai: số thực nhị phân có sai số. **Đúng:** làm việc hoàn toàn trên số nguyên (tử/mẫu) hoặc `Fraction`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "phân số", "tối giản", biểu thức chuỗi -> số học số nguyên + `gcd`, không dùng `float`.
- **Mẫu số chung cố định.** Khi các mẫu số bị chặn bé (đây `<= 10`), lấy `lcm` của tất cả giá trị có thể — một hằng số — để chuyển bài toán về cộng số nguyên.
- **Mẫu số không bị chặn?** Dùng Cách 2 (rút gọn từng bước) hoặc `Fraction`.
- **Mẹo parse:** `replace('-', '+-').split('+')` là cách nhanh nhất để tách biểu thức cộng-trừ có dấu. Cũng dùng được trong "Basic Calculator" đơn giản.
- **`gcd` với số âm / 0:** `math.gcd` luôn không âm và `gcd(0, n) = n`, nên công thức `x//g, y//g` đúng cho mọi dấu và cho 0.
- Chia `//` an toàn khi chia **hết**; luôn kiểm tra điều đó trước khi dùng với số âm.
- Với Java/C++, chú ý tràn số khi nhân tất cả mẫu số; dùng mẫu số cố định hoặc `long` + rút gọn từng bước.
- Tự test nhanh: kết quả 0, số nguyên (`n/1`), phân số âm, số hai chữ số.

## 15. Biến thể

1. **Mẫu số tùy ý (không bị chặn bởi 10).** Cách 3 mất hiệu lực; Cách 2 (rút gọn từng bước) vẫn đúng và an toàn vì Python có số nguyên lớn.
2. **Đổi kết quả sang hỗn số** (mixed number): `7/3 -> "2 1/3"`.
3. **Cộng nhiều phân số và chia cho `k`** (trung bình cộng) — bài tập 5.
4. **Phép nhân/chia phân số** hoặc có ngoặc: cần bộ phân tích cú pháp (parser) hoặc stack, như bài Basic Calculator.
5. Bài liên quan: Fraction to Recurring Decimal (166), Basic Calculator II (227).

Dưới đây là code cho biến thể 1 và 2, kiểm bằng oracle.

In [30]:
def fraction_general(expression: str) -> str:
    """Mẫu số/tử bất kỳ: rút gọn từng bước (như Cách 2) nhưng tách bằng regex."""
    x, y = 0, 1
    for term in re.findall(r"[+-]?\d+/\d+", expression):
        a, b = map(int, term.split("/"))
        x = x * b + a * y
        y *= b
        g = gcd(x, y)
        x //= g
        y //= g
    return f"{x}/{y}"

def to_mixed(frac: str) -> str:
    n, d = map(int, frac.split("/"))
    whole, rem = divmod(abs(n), d)
    sign = "-" if n < 0 else ""
    if rem == 0:
        return f"{sign}{whole}"
    if whole == 0:
        return f"{sign}{rem}/{d}"
    return f"{sign}{whole} {rem}/{d}"

big = "1/999983-1/1000003+123456/7+99/100000"
print("mẫu số lớn:", fraction_general(big), "| oracle:", oracle(big))
assert fraction_general(big) == oracle(big)
print(to_mixed("7/3"), "|", to_mixed("-7/3"), "|", to_mixed("1/3"), "|", to_mixed("6/3"), "|", to_mixed("0/1"))
assert to_mixed("7/3") == "2 1/3" and to_mixed("-7/3") == "-2 1/3" and to_mixed("6/3") == "2" and to_mixed("0/1") == "0"

mẫu số lớn: 12345427853960686364657/699990199964300000 | oracle: 12345427853960686364657/699990199964300000
2 1/3 | -2 1/3 | 1/3 | 2 | 0


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Nhân hết mẫu, rút gọn cuối | Công thức cộng thô | `O(L)` | `O(L)` | Chỉ trong Python; hiểu công thức. Dễ tràn số ở ngôn ngữ khác |
| 2. Quét tay + rút gọn từng bước | Bất biến "tổng đã tối giản" | `O(L)` | `O(1)` | Mẫu số không bị chặn; không dùng bộ nhớ phụ |
| 3. Mẫu số cố định 2520 | `lcm(1..10)` -> cộng số nguyên | `O(L)` | `O(L)` | **Cách nộp chính**: gọn nhất, an toàn nhất |
| 4. `Fraction` | Thư viện lo hết | `O(L)` | `O(L)` | Viết nhanh khi thư viện được phép |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(gcd(-12, 18))
print("-1/2+1/3".replace("-", "+-").split("+"))
print(-7 // 2)
```

In [31]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `expression = "1/2-1/3+1/6"` và mẫu chung `2520`: tính `total` (tổng các tử khi quy về đơn vị `1/2520`) và chuỗi đáp án sau rút gọn.

In [32]:
# Bài 2: điền kết quả tính tay
hand_total = None       # tổng các tử trên mẫu 2520
hand_answer = None      # chuỗi dạng "tử/mẫu"

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo Cách 3.

In [33]:
def exercise_3_fixed(expression: str) -> str:
    base = 2520
    total = 0
    # TODO 1: replace('-', '+-') rồi split('+') để có các hạng tử đã mang dấu
    # TODO 2: bỏ qua hạng tử rỗng
    # TODO 3: tách tử/mẫu bằng split('/'), cộng int(a) * (base // int(b)) vào total
    # TODO 4: rút gọn bằng gcd và trả về f"{tử}/{mẫu}"
    return None


# assert exercise_3_fixed("-1/2+1/2+1/3") == "1/3"
# assert exercise_3_fixed("1/3-1/2") == "-1/6"
# assert exercise_3_fixed("10/3-7/10") == "79/30"

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, hãy tự viết `exercise_4_running(expression)`: giữ tổng `x/y`, đọc từng hạng tử và **rút gọn sau mỗi bước** (bạn tự chọn cách đọc chuỗi: quét tay, `split` hoặc `re`).

In [34]:
def exercise_4_running(expression: str) -> str:
    # TODO
    pass


# assert exercise_4_running("-1/2+1/2+1/3") == "1/3"
# assert exercise_4_running("1/3-1/2") == "-1/6"
# assert exercise_4_running("-1/2+1/2") == "0/1"
# assert exercise_4_running("5/1") == "5/1"

### Bài 5 — Mini challenge: trung bình cộng các phân số (khó)

Cho cùng dạng biểu thức; trả về **trung bình cộng** của các hạng tử (tổng chia số hạng tử) dưới dạng phân số tối giản `"tử/mẫu"`. Chú ý: mẫu của kết quả có thể vượt 2520 (vì chia thêm cho `k <= 10`).

```python
exercise_5_average("1/2+1/2")         -> "1/2"      (tổng 1, 2 hạng tử)
exercise_5_average("-1/2+1/2")        -> "0/1"
exercise_5_average("1/3+1/6+1/2")     -> "1/3"      (tổng 1, 3 hạng tử)
exercise_5_average("5/1")             -> "5/1"
```
Gợi ý: đếm số hạng tử `k`; mẫu của tổng nhân với `k` rồi rút gọn.

In [35]:
def exercise_5_average(expression: str) -> str:
    # TODO
    pass


# assert exercise_5_average("1/2+1/2") == "1/2"
# assert exercise_5_average("-1/2+1/2") == "0/1"
# assert exercise_5_average("1/3+1/6+1/2") == "1/3"
# assert exercise_5_average("5/1") == "5/1"

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `math.gcd` không bao giờ trả số âm.
- Mức 2: chuỗi bắt đầu bằng `-1/2`, sau `replace` thành `+-1/2+1/3`; tách theo `+` có phần tử đầu rỗng.
- Mức 3: kết quả `6`, `['', '-1/2', '1/3']`, `-4` (làm tròn xuống).

**Bài 2**
- Mức 1: `2520 // 2 = 1260`, `2520 // 3 = 840`, `2520 // 6 = 420`.
- Mức 2: `total = 1260 - 840 + 420`.
- Mức 3: `total = 840`, `gcd(840, 2520) = 840` -> `"1/3"`.

**Bài 3**
- Mức 1: `for token in expression.replace("-", "+-").split("+")`.
- Mức 2: `if not token: continue`; `a, b = token.split("/")`.
- Mức 3: `total += int(a) * (base // int(b))`; cuối `g = gcd(total, base)`; trả `f"{total // g}/{base // g}"`.

**Bài 4**
- Mức 1: `x, y = 0, 1`.
- Mức 2: với mỗi `a/b` có dấu: `x, y = x * b + a * y, y * b`.
- Mức 3: ngay sau mỗi lần cộng `g = gcd(x, y)`; `x //= g`; `y //= g`; cuối trả `f"{x}/{y}"`.

**Bài 5**
- Mức 1: dùng lại cách tính tổng, thêm biến đếm `k`.
- Mức 2: trung bình = `total / (base * k)`.
- Mức 3: `g = gcd(total, base * k)`; trả `f"{total // g}/{base * k // g}"` (mẫu luôn dương nên dấu nằm ở tử).

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [36]:
print(gcd(-12, 18))
print("-1/2+1/3".replace("-", "+-").split("+"))
print(-7 // 2)

assert gcd(-12, 18) == 6
assert "-1/2+1/3".replace("-", "+-").split("+") == ["", "-1/2", "1/3"]
assert -7 // 2 == -4

6
['', '-1/2', '1/3']
-4


### Đáp án Bài 2

`1/2 - 1/3 + 1/6` trên mẫu 2520: `1260 - 840 + 420 = 840`. `gcd(840, 2520) = 840` -> `1/3`.

In [37]:
hand_total_solution = 1260 - 840 + 420
hand_answer_solution = "1/3"
assert hand_total_solution == 840
g = gcd(hand_total_solution, 2520)
assert f"{hand_total_solution // g}/{2520 // g}" == hand_answer_solution == fraction_fixed("1/2-1/3+1/6") == oracle("1/2-1/3+1/6")
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [38]:
def exercise_3_fixed_solution(expression: str) -> str:
    base = 2520
    total = 0
    for token in expression.replace("-", "+-").split("+"):
        if not token:
            continue
        a, b = token.split("/")
        total += int(a) * (base // int(b))
    g = gcd(total, base)
    return f"{total // g}/{base // g}"

assert exercise_3_fixed_solution("-1/2+1/2+1/3") == "1/3"
assert exercise_3_fixed_solution("1/3-1/2") == "-1/6"
assert exercise_3_fixed_solution("10/3-7/10") == "79/30"
random.seed(33)
for _ in range(300):
    e = make_expr([(random.choice((1, -1)), *random.choice(FR)) for _ in range(random.randint(1, 10))])
    assert exercise_3_fixed_solution(e) == oracle(e)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [39]:
def exercise_4_running_solution(expression: str) -> str:
    x, y = 0, 1
    for term in re.findall(r"[+-]?\d+/\d+", expression):
        a, b = map(int, term.split("/"))
        x, y = x * b + a * y, y * b
        g = gcd(x, y)
        x //= g
        y //= g
    return f"{x}/{y}"

assert exercise_4_running_solution("-1/2+1/2+1/3") == "1/3"
assert exercise_4_running_solution("1/3-1/2") == "-1/6"
assert exercise_4_running_solution("-1/2+1/2") == "0/1"
assert exercise_4_running_solution("5/1") == "5/1"
random.seed(44)
for _ in range(300):
    e = make_expr([(random.choice((1, -1)), *random.choice(FR)) for _ in range(random.randint(1, 10))])
    assert exercise_4_running_solution(e) == oracle(e)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [40]:
def exercise_5_average_solution(expression: str) -> str:
    base = 2520
    total = 0
    k = 0
    for token in expression.replace("-", "+-").split("+"):
        if not token:
            continue
        a, b = token.split("/")
        total += int(a) * (base // int(b))
        k += 1
    den = base * k
    g = gcd(total, den)
    return f"{total // g}/{den // g}"

def average_oracle(expression: str) -> str:
    terms = re.findall(r"[+-]?\d+/\d+", expression)
    s = sum((Fraction(int(t.split("/")[0]), int(t.split("/")[1])) for t in terms), Fraction(0))
    v = s / len(terms)
    return f"{v.numerator}/{v.denominator}"

assert exercise_5_average_solution("1/2+1/2") == "1/2"
assert exercise_5_average_solution("-1/2+1/2") == "0/1"
assert exercise_5_average_solution("1/3+1/6+1/2") == "1/3"
assert exercise_5_average_solution("5/1") == "5/1"
random.seed(55)
for _ in range(500):
    e = make_expr([(random.choice((1, -1)), *random.choice(FR)) for _ in range(random.randint(1, 10))])
    assert exercise_5_average_solution(e) == average_oracle(e), e
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (mẫu số cố định + gcd)

```python
class Solution:
    def fractionAddition(self, expression: str) -> str:
        from math import gcd
        base = 2520                                   # lcm(1..10)
        total = 0
        for token in expression.replace("-", "+-").split("+"):
            if not token:
                continue
            a, b = token.split("/")
            total += int(a) * (base // int(b))
        g = gcd(total, base)
        return f"{total // g}/{base // g}"
```

Công thức nhanh: `a/b = a*(D/b)/D`, cộng các tử, rút gọn bằng `gcd`; `0` -> `0/1`, số nguyên -> `n/1`.

### Checklist trước khi submit

- [ ] Có xử lý số hai chữ số (`10`) và hạng tử đầu không có dấu?
- [ ] Dấu `-` được giữ lại (dùng `replace('-', '+-')`, không dùng `split('-')`)?
- [ ] Bỏ qua phần tử rỗng sau khi tách?
- [ ] Chỉ dùng `//` trên phép chia hết, không dùng `/` (tránh `float`)?
- [ ] Mẫu số cố định chia hết mọi mẫu có thể (`2520` cho 1..10)?
- [ ] Rút gọn bằng `gcd` cuối cùng; kết quả 0 ra `0/1`, số nguyên ra `n/1`?
- [ ] Đã thử `"1/3"`, `"-1/2+1/2"`, `"10/3-7/10"`, mười hạng tử `10/1`?
- [ ] Tên method đúng `fractionAddition(self, expression)` và trả `str`?